# Transformer Linear Layer Performance Predictor

## Fresh GPU experiment — v1.1.0

Select **Runtime > Change runtime type > GPU**, then **Runtime > Run all**.
Authorize Google Drive when prompted. Verified evidence checkpoints are saved
to your Drive before and during the experiment; active measurement files stay
on the Colab VM. Saving the notebook alone does not save those measurements.
This notebook contains the complete source. No previous ZIP, timing CSV,
trained predictor, model weights, or repository checkout is needed.

The default experiment performs:

1. Fresh FP16/BF16 collection on the **609-shape** model-derived workload.
2. Training-only grouped cross-validation, configuration/model selection,
   full-training refit, and separate test/challenge/stress evaluation.
3. A deterministic **77-shape repeat** on the same GPU, without refitting.
4. Repeat comparison and one combined report with raw-record and timing-quality checks.
5. One complete result ZIP, also copied and verified in Google Drive. Earlier
   checkpoints retain evidence if the runtime disappears before the final export.

Every split is measured before fitting. Test labels may exist on disk, but
never enter fitting or model selection. There is no freeze-before-measurement step.

The primary target is preallocated `Y = X @ W.T` under CUDA Graph replay.
Warm reuse and eligible four-buffer rotation are separate contexts. GPU memory
reads/writes are inside the interval; host transfers, graph capture and serving
queues are outside it. Rotation does not guarantee cold-cache execution.


In [ ]:
from datetime import datetime, timezone
import uuid

EXPERIMENT_ID = "fresh_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S") + "_" + uuid.uuid4().hex[:6]
PROFILE = "standard"                 # 609 shapes/dtype; "smoke" is a 32-shape software/hardware check.
DTYPES = ("float16", "bfloat16")       # Both interleaved; each precision/cache context is modeled separately.
SIMULATE = False                      # Keep False for actual GPU data.
INCLUDE_REPEAT = True                 # Required by the default complete experiment; 77/609 shapes.
INCLUDE_LAYER = False                 # Optional random-weight decoder-layer hybrid estimate.
SKIP_PROFILES = False                 # Separate diagnostic profiling after timing.
MAX_WALL_SECONDS = 7200               # Maximum per collection stage, not a predicted duration.
PIN_ANALYSIS_DEPS = True

_collection_stages = 2 if INCLUDE_REPEAT else 1
_worker_limit = MAX_WALL_SECONDS * _collection_stages + 900 + (900 if INCLUDE_LAYER else 0)
print("Experiment:", EXPERIMENT_ID)
print("Collection stages:", _collection_stages, "| Per-stage cap:", MAX_WALL_SECONDS, "seconds")
print("Combined worker cap:", _worker_limit, "seconds; setup and download are additional.")
print("These are upper limits, not runtime or cost predictions.")


## Mount backup storage

Authorize your Google Drive connection. Backups are stored under
`MyDrive/Transformer_Linear_Layer_Performance_Predictor/experiments/`, in a
new folder for this experiment. Existing experiment folders are never reused.

The first verified checkpoint is required before any GPU measurements. Later
checkpoints run synchronously between complete timing visits, before profiling,
and after major stages. They are outside CUDA event timing intervals. Their
pauses are recorded and count toward the collection wall-time budget; they can
affect thermal state, so controls and timing-quality checks remain necessary.

A backup is confirmed only after file-copy readback verification. Drive errors
stop the required checkpoint path; no automatic measurement retry occurs.


In [ ]:
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
_my_drive = Path("/content/drive/MyDrive")
if not _my_drive.is_dir():
    raise RuntimeError("Google Drive is not mounted. Complete authorization before measuring.")
BACKUP_DIRECTORY = _my_drive / "Transformer_Linear_Layer_Performance_Predictor" / "experiments"
BACKUP_DIRECTORY.mkdir(parents=True, exist_ok=True)
print("Backup directory:", BACKUP_DIRECTORY)
print("The experiment verifies its initial backup before starting GPU measurement.")


## Load the included experiment

This verifies the embedded source and compiles Python files without measuring
GPU work. The payload contains no historical measured result files. Its analysis
stack may be installed; the runtime's existing PyTorch/CUDA is retained.


In [ ]:
# Verify and load the embedded release; this cell performs no GPU measurement.
import base64, hashlib, types
_LAUNCHER_B64 = '...'
_launcher_source = base64.b64decode(_LAUNCHER_B64).decode('utf-8')
assert hashlib.sha256(_launcher_source.encode()).hexdigest() == 'f24d6399ebdf53c6e81a0a81b186b25be8c467a29ed1e22cf54163dc85fcc321', 'Launcher integrity mismatch'
launcher = types.ModuleType('linear_perf_colab')
exec(compile(_launcher_source, 'linear_perf_colab.py', 'exec'), launcher.__dict__)
print(launcher.self_check())


## Run the complete experiment

Progress reports primary collection/analysis, repeat collection, and comparison.
The 7,200-second cap applies separately to primary and repeat collection, so the
complete experiment can exceed two hours. Analysis and export add time.

Failures are retained and exported when the process can finish. A verified
checkpoint before profiling protects completed timing records from a later
runtime loss. Profiling reports progress separately; 609/609 timing shapes does
not yet mean that model evaluation and the short repeat have completed.

An abrupt loss before a checkpoint finishes can still lose the latest work.
Checkpoints are preserved evidence, not automatic resume points or proof of a
complete experiment. Never silently append measurements from a replacement GPU.


In [ ]:
result = launcher.run_experiment(
    experiment_id=EXPERIMENT_ID, profile=PROFILE, dtypes=DTYPES,
    simulate=SIMULATE, include_repeat=INCLUDE_REPEAT, include_layer=INCLUDE_LAYER,
    skip_profiles=SKIP_PROFILES, max_wall_seconds=MAX_WALL_SECONDS,
    pin_analysis_deps=PIN_ANALYSIS_DEPS, backup_directory=BACKUP_DIRECTORY,
)
print("Launcher status:", result["status"])
print("Experiment status:", result.get("experiment_status", "Inspect launcher_status.json"))
print("Result archive:", result["archive"])
print("Final backup status:", result.get("backup_status", "unknown"))
if result.get("backup_archive"):
    print("Saved backup:", result["backup_archive"])
if result.get("backup_status") == "failed":
    print("Download the local archive immediately. Final Drive copy failed:", result.get("backup_error"))
if result["status"] != "worker_completed":
    print("Download the archive below. Completed primary results may still be available; inspect stage statuses.")


## Download the single complete result

After extraction, open **`run/report.html`** first. It links to the primary
prediction report, repeat comparison, selected-model metrics and quality records.
Raw data are under `run/primary/dataset/` and `run/repeat/dataset/`.

If the download does not start, rerun only the download cell. It can also find
saved archives when the in-memory `result` variable is unavailable: mount Drive
first, then run this cell. If multiple experiments exist, select the exact ID
printed below. A final archive is preferred; otherwise a clearly labeled
checkpoint is offered. Re-running the settings and experiment cells starts a
new measurement attempt.

After losing the entire runtime, the separate **Download_Saved_Results.ipynb**
can retrieve saved evidence on CPU without starting another experiment.


In [ ]:
"""Read and verify saved experiment archives without the original notebook state.

This standalone helper performs no measurement, fitting, extraction, or source
execution. It can be embedded in a separate CPU-only recovery notebook.
"""
from __future__ import annotations

import hashlib
import json
from pathlib import Path
import re
import zipfile


DEFAULT_BACKUP_DIRECTORY = "/content/drive/MyDrive/Transformer_Linear_Layer_Performance_Predictor/experiments"


def _file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _valid_id(value):
    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", value):
        raise ValueError("Experiment ID must contain only letters, numbers, underscores, and hyphens")
    return value


def list_saved_experiments(backup_directory):
    root = Path(backup_directory).expanduser()
    if not root.is_dir():
        raise FileNotFoundError("Backup folder is unavailable. Mount Google Drive first, or check BACKUP_DIRECTORY.")
    experiments = []
    for folder in sorted(root.glob("linear-perf_*")):
        if not folder.is_dir() or folder.is_symlink():
            continue
        experiment_id = folder.name.removeprefix("linear-perf_")
        if not re.fullmatch(r"[A-Za-z0-9][A-Za-z0-9_-]{0,63}", experiment_id):
            continue
        archives = [p.name for p in sorted(folder.glob("*.zip"))
                    if p.is_file() and not p.is_symlink() and p.with_name(p.name + ".sha256").is_file()]
        experiments.append({"experiment_id": experiment_id, "directory": str(folder),
                            "archives": archives, "archive_count": len(archives)})
    return experiments


def _inspect_saved_archive(path, expected_sha256=None):
    path = Path(path)
    if not path.is_file() or path.is_symlink():
        raise FileNotFoundError("The selected archive is unavailable or is a link: " + str(path))
    if expected_sha256 is None:
        sidecar = path.with_name(path.name + ".sha256")
        if not sidecar.is_file() or sidecar.is_symlink():
            raise ValueError("No completed checksum receipt exists for this archive: " + path.name)
        fields = sidecar.read_text(encoding="ascii").strip().split(maxsplit=1)
        if len(fields) != 2 or fields[1].lstrip("*") != path.name:
            raise ValueError("Archive checksum receipt names a different file")
        expected_sha256 = fields[0]
    if not isinstance(expected_sha256, str) or not re.fullmatch(r"[0-9a-f]{64}", expected_sha256):
        raise ValueError("Invalid SHA256 receipt")
    actual = _file_sha256(path)
    if actual != expected_sha256:
        raise ValueError("Archive checksum mismatch. Preserve this file; select another verified backup.")
    with zipfile.ZipFile(path) as archive:
        names = set(archive.namelist())
        def small_json(name):
            if name not in names:
                return {}
            if archive.getinfo(name).file_size > 4 * 1024 * 1024:
                raise ValueError("Archive metadata exceeds the inspection limit")
            value = json.loads(archive.read(name))
            if not isinstance(value, dict):
                raise ValueError("Archive metadata must be a JSON object")
            return value
        checkpoint = small_json("checkpoint.json")
        run = small_json("run/status.json") or small_json("run/summary.json")
        launch = small_json("launcher_status.json")
    kind = "checkpoint" if checkpoint else "results"
    status = run.get("status", "unknown")
    return {"path": str(path), "sha256": actual, "kind": kind, "status": status,
            "bytes": path.stat().st_size, "launcher_status": launch.get("status", "unknown"),
            "checkpoint": checkpoint, "measurement_kind": run.get("measurement_kind", "unknown"),
            "all_requested_stages_completed": run.get("all_requested_stages_completed", False),
            "note": ("This is a preserved checkpoint, not proof of a completed experiment. No work was resumed."
                     if kind == "checkpoint" else
                     "An exported archive can contain a failed or incomplete run. Inspect its recorded status.")}


def locate_saved_result(backup_directory, *, experiment_id="", archive_name="", local_result=None):
    """Select one existing archive and verify it before offering a download.

    An explicit experiment/filename takes precedence. Without a selection, a
    surviving launcher result can resolve the local archive; otherwise exactly
    one saved experiment is required. No arbitrary latest-experiment selection.
    """
    if experiment_id:
        _valid_id(experiment_id)
    if archive_name and (Path(archive_name).name != archive_name or "\\" in archive_name
                         or not archive_name.endswith(".zip")):
        raise ValueError("ARCHIVE_NAME must be a ZIP filename, not a path")
    if not experiment_id and not archive_name and isinstance(local_result, dict):
        local = local_result.get("archive")
        expected = local_result.get("result_archive_sha256")
        if local and expected and Path(local).is_file():
            return _inspect_saved_archive(local, expected)
    experiments = list_saved_experiments(backup_directory)
    if experiment_id:
        experiments = [item for item in experiments if item["experiment_id"] == experiment_id]
        if not experiments:
            raise FileNotFoundError("No saved folder for the selected experiment: " + experiment_id)
    if len(experiments) != 1:
        ids = [item["experiment_id"] for item in experiments]
        raise ValueError("Set RECOVERY_EXPERIMENT_ID to one exact listed ID. Available IDs: " + repr(ids))
    experiment = experiments[0]
    folder = Path(experiment["directory"])
    experiment_id = experiment["experiment_id"]
    if archive_name:
        selected = folder / archive_name
    else:
        final = f"linear-perf_{experiment_id}_RESULTS.zip"
        if final in experiment["archives"]:
            selected = folder / final
        else:
            prefix = f"linear-perf_{experiment_id}_CHECKPOINT_"
            checkpoints = [name for name in experiment["archives"] if name.startswith(prefix)]
            if not checkpoints:
                raise FileNotFoundError("No completed archive/checksum pair is saved for this experiment yet")
            selected = folder / sorted(checkpoints)[-1]
    return _inspect_saved_archive(selected)


from google.colab import files

# Leave these empty for the current result, or for the only saved experiment.
# If multiple experiments are saved, enter one exact ID from the printed list.
RECOVERY_EXPERIMENT_ID = ""
RECOVERY_ARCHIVE_NAME = ""
_backup_root = globals().get("BACKUP_DIRECTORY", DEFAULT_BACKUP_DIRECTORY)
_current_result = globals().get("result")
if not isinstance(_current_result, dict):
    print("Saved experiments:")
    for item in list_saved_experiments(_backup_root):
        print(item["experiment_id"], "|", item["archive_count"], "archive(s)")

saved_result = locate_saved_result(
    _backup_root, experiment_id=RECOVERY_EXPERIMENT_ID,
    archive_name=RECOVERY_ARCHIVE_NAME, local_result=_current_result,
)
print("Archive:", saved_result["path"])
print("Kind:", saved_result["kind"], "| Recorded status:", saved_result["status"])
print(saved_result["note"])
print("Archive SHA256:", saved_result["sha256"])
files.download(saved_result["path"])


## How to interpret this run

- A completed run can still have timing-quality qualifications. Inspect controls,
  drift and unstable rows alongside mean, median, P90 and worst-case errors.
- CV scores select a model; the held-out scores evaluate that selection separately.
- The repeat subset checks stability on known shapes. It does not create a second
  independent new-shape test set and it never refits the source model.
- Compare model families on the same new test data and measurement context.
  An aggregate over this workload is not directly comparable to a different old workload.
- If `INCLUDE_LAYER = True`, the optional layer stage uses predicted linear GEMM
  costs plus independently measured other-operation costs. It is a hybrid estimate,
  not a fully predicted LLM/serving benchmark.
- Another GPU is an additional experiment; this single-GPU run does not establish
  cross-architecture accuracy. No GPU is rented automatically.

The source and result metadata retain release **1.1.0** plus exact source hashes.
CPU simulation is explicitly labeled and never presented as GPU evidence.
